<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/etapa-2-grafos/11%20-%20Modelagem%20da%20Tubulacao%20e%20Instrumentos%20como%20Grafo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 11 - Notebook: Modelagem Topológica de Tubulações da Linha de Envase como Dígrafos Ponderados

Neste notebook implementamos a classe base `GrafoTubulacao` para representar a malha hidráulica e de dosagem da **Linha de Envasamento de Bebidas (SCADA-Core - Grupo 3)** como um Grafo Dirigido e Ponderado $G=(V, E, W)$.


In [1]:
from typing import List, Dict, Tuple, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz: List[List[float]], rotulos_linhas: List[str], rotulos_cols: List[str]) -> str:
    """Formata matriz 2D em tabela ASCII."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoTubulacao:
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)

        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0

        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_tubulacao(self, origem: str, destino: str, comprimento_m: float,
                           tag_valvula: str, diametro_pol: float = 3.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]

        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m

        self.arestas_detalhes.append({
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Válvula ISA": tag_valvula,
            "Diâmetro (pol)": diametro_pol
        })

    def obter_graus(self) -> List[Dict[str, Any]]:
        graus = []
        for i, v in enumerate(self.vertices):
            deg_out = sum(self.adj_binaria[i])
            deg_in = sum(self.adj_binaria[r][i] for r in range(self.n))
            graus.append({"Componente / Nó": v, "Grau Entrada (deg-)": deg_in, "Grau Saída (deg+)": deg_out})
        return graus

# Vértices da Linha de Envasamento e Dosagem (Grupo 3)
nos_envase = [
    "TS1_Suprimento",
    "VS1_Succao",
    "BC1_Bomba",
    "AS1_Acumulador",
    "VS2_Envase",
    "SQ2_Medicao",
    "EST_Envase",
    "VALV_Alivio"
]

rede = GrafoTubulacao(nos_envase)

# Adição dos trechos hidráulicos e de dosagem da planta modelo
rede.adicionar_tubulacao("TS1_Suprimento", "VS1_Succao", 5.0, "VS1", 3.0)
rede.adicionar_tubulacao("VS1_Succao", "BC1_Bomba", 3.0, "SP1_SQ1", 3.0)
rede.adicionar_tubulacao("BC1_Bomba", "AS1_Acumulador", 8.0, "CHECK_V", 2.5)
rede.adicionar_tubulacao("AS1_Acumulador", "VS2_Envase", 10.0, "VS2", 2.0)
rede.adicionar_tubulacao("AS1_Acumulador", "VALV_Alivio", 6.0, "SP2_VS3", 2.0)
rede.adicionar_tubulacao("VALV_Alivio", "TS1_Suprimento", 12.0, "RET_V", 2.0)
rede.adicionar_tubulacao("VS2_Envase", "SQ2_Medicao", 2.0, "SQ2_IN", 1.5)
rede.adicionar_tubulacao("SQ2_Medicao", "EST_Envase", 1.5, "BICO_FILL", 1.5)
rede.adicionar_tubulacao("AS1_Acumulador", "SQ2_Medicao", 11.0, "XV_BYPASS", 2.0)

print("Tabela de Tubulações da Linha de Envase:")
print(formatar_tabela(rede.arestas_detalhes))
print("\n--- Graus Topológicos dos Componentes ---")
print(formatar_tabela(rede.obter_graus()))
print("\n--- Matriz de Adjacência Ponderada (Distância em Metros) ---")
print(formatar_matriz(rede.adj_pesos, rede.vertices, rede.vertices))


Tabela de Tubulações da Linha de Envase:
Origem         | Destino        | Comprimento (m) | Válvula ISA | Diâmetro (pol)
---------------+----------------+-----------------+-------------+---------------
TS1_Suprimento | VS1_Succao     | 5.0             | VS1         | 3.0           
VS1_Succao     | BC1_Bomba      | 3.0             | SP1_SQ1     | 3.0           
BC1_Bomba      | AS1_Acumulador | 8.0             | CHECK_V     | 2.5           
AS1_Acumulador | VS2_Envase     | 10.0            | VS2         | 2.0           
AS1_Acumulador | VALV_Alivio    | 6.0             | SP2_VS3     | 2.0           
VALV_Alivio    | TS1_Suprimento | 12.0            | RET_V       | 2.0           
VS2_Envase     | SQ2_Medicao    | 2.0             | SQ2_IN      | 1.5           
SQ2_Medicao    | EST_Envase     | 1.5             | BICO_FILL   | 1.5           
AS1_Acumulador | SQ2_Medicao    | 11.0            | XV_BYPASS   | 2.0           

--- Graus Topológicos dos Componentes ---
Componente / Nó | Grau En